# GalMorphAI · 01 — Preprocesamiento del piloto

Notebook editable para **Google Colab**, documentado en español. Procesa exclusivamente un piloto de **10 galaxias**, con representación de los códigos **0, 2, 4, 8, 12, 16**, sin inferir su semántica. No entrena modelos ni ejecuta una descarga masiva.

**Entrada:** `gs://galmorphai-science/metadata/catalogs/dataset_ring_clean.csv`  
**Proyecto de autenticación y cliente:** `brave-server-511721-p7`  
**Originales:** `gs://galmorphai-science/raw/legacy_surveys/pilot/`

Los cutouts son productos de imagen del visor de Legacy Surveys, no exposiciones crudas del telescopio. Aquí «original» significa los bytes FITS recibidos, conservados sin modificar. El preprocesamiento de esta etapa comprende selección, descarga, diagnóstico y visualización; no rellena NaN, normaliza los FITS ni elimina objetos por calidad.

## Uso rápido
1. Sube este archivo a [Google Colab](https://colab.research.google.com/), con entorno Python 3 y CPU.
2. Revisa los parámetros; si conoces los nombres del CSV, configura `COLUMNAS`. RA/DEC deben estar en grados ICRS.
3. Ejecuta las celdas en orden y autoriza la cuenta Google con acceso al proyecto y al bucket.
4. Revisa la tabla seleccionada. La celda de procesamiento solicita únicamente esos diez objetos.
5. Para reanudar, ejecuta de nuevo con los mismos parámetros y catálogo. Los objetos ya presentes en GCS se recuperan y verifican; no se reemplazan.

Se requieren permisos de lectura y listado de objetos, y creación de objetos en los prefijos de salida. No se requieren claves privadas en el notebook ni permisos para borrar objetos. Los cargos de almacenamiento y red dependen de tu configuración de GCP.


In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'numpy>=1.26,<3', 'pandas>=2.2,<3', 'astropy>=6,<8',
    'matplotlib>=3.8,<4', 'requests>=2.32,<3',
    'google-cloud-storage>=2.18,<4'])


0

**Explicación.** Instala dependencias compatibles en el entorno temporal de Colab. Se registran las versiones efectivamente utilizadas en cada ejecución; los rangos no constituyen un entorno congelado. Si Colab pide reiniciar tras la instalación, reinicia y continúa desde la siguiente celda.


In [2]:
import io, json, hashlib, time, uuid, warnings, platform
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u
from google.cloud import storage
from google.api_core.exceptions import PreconditionFailed

PROYECTO = 'brave-server-511721-p7'
BUCKET = 'galmorphai-science'
CATALOGO = 'metadata/catalogs/dataset_ring_clean.csv'
PREFIJO_RAW = 'raw/legacy_surveys/pilot/'
PREFIJO_META = 'metadata/legacy_surveys/pilot/'
COLUMNAS = {'ra': 'ra', 'dec': 'dec', 'label': 'anillos'}  # Nombres exactos del catálogo
CSV_SEPARADOR = ','
ETIQUETAS = [0, 2, 4, 8, 12, 16]
N_PILOTO = 10
SEMILLA = 42
CAPA = 'ls-dr10'  # Capa fijada expresamente; no usar una referencia móvil.
BANDAS = 'grz'
TAMANO = 256  # Píxeles por lado; límite conservador documentado: 512.
ESCALA = 0.262  # Segundos de arco por píxel; campo: ~67 segundos de arco.
ENDPOINT = 'https://www.legacysurvey.org/viewer/fits-cutout'
INTENTOS_HTTP = 3
TIMEOUT = (15, 120)  # Conexión y lectura, segundos.
PAUSA_SEGUNDOS = 1.0
MAX_BYTES = 25 * 1024 * 1024
MAX_FRACCION_NO_FINITA = 0.05
TOL_CENTRO_ARCSEC = 2.0
DIRECTORIO = Path('/content/galmorphai_pilot')
DIRECTORIO.mkdir(parents=True, exist_ok=True)
assert N_PILOTO == 10, 'Este notebook está limitado a diez galaxias.'
assert ETIQUETAS == [0, 2, 4, 8, 12, 16] and BANDAS == 'grz'
assert 1 <= TAMANO <= 512 and ESCALA > 0
assert 1 <= INTENTOS_HTTP <= 5 and 0 <= MAX_FRACCION_NO_FINITA <= 1

def ahora():
    return datetime.now(timezone.utc).isoformat()

def sha(data):
    return hashlib.sha256(data).hexdigest()

def serializar(obj):
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, allow_nan=False).encode('utf-8')


**Explicación.** Centraliza los parámetros científicos y operativos. Las seis etiquetas se tratan únicamente como códigos. Los límites impiden ampliar por accidente el tamaño del piloto. El tamaño angular fijo puede cortar galaxias grandes: conviene revisarlo visualmente antes de cualquier fase posterior. Los umbrales de calidad son heurísticos y configurables, no una clasificación morfológica.


In [3]:
from google.colab import auth
import google.auth

auth.authenticate_user()
credenciales, _ = google.auth.default()
cliente = storage.Client(project=PROYECTO, credentials=credenciales)
bucket = cliente.bucket(BUCKET)
blob_catalogo = bucket.get_blob(CATALOGO)
if blob_catalogo is None:
    raise FileNotFoundError(f'No existe gs://{BUCKET}/{CATALOGO}')
generacion_catalogo = int(blob_catalogo.generation)
contenido_catalogo = blob_catalogo.download_as_bytes(
    if_generation_match=generacion_catalogo, timeout=120)
sha_catalogo = sha(contenido_catalogo)
catalogo = pd.read_csv(io.BytesIO(contenido_catalogo), sep=CSV_SEPARADOR)
print('Proyecto:', cliente.project, '| Filas:', len(catalogo))
print('Columnas disponibles:', catalogo.columns.tolist())
print('Generación del catálogo:', generacion_catalogo, '| SHA-256:', sha_catalogo)
display(catalogo.head())


Proyecto: brave-server-511721-p7 | Filas: 8527
Columnas disponibles: ['objID', 'ra', 'dec', 'z', 'anillos']
Generación del catálogo: 1792107309119574 | SHA-256: 7fbe2c411ecd0d0ad90e32e524d9bc3e45f874806123b99d7d43658db8a55ed6


,objID,ra,dec,z,anillos
0,1237648721210769659,134.44717,-0.199973,0.028206,0.0
1,1237648705657307354,198.23356,0.941188,0.048037,0.0
2,1237648705120895059,199.29492,0.527571,0.024114,0.0
3,1237648720150724863,165.74061,-0.962095,0.033483,0.0
4,1237649919509594232,31.37202,13.251016,0.024694,0.0


**Explicación.** Autentica interactivamente en Colab y construye el cliente para el proyecto indicado. La cuenta elegida debe tener acceso al bucket, que puede pertenecer a un proyecto distinto. La lectura fija la generación de GCS: si cambia mientras se descarga, la operación falla en vez de mezclar versiones. No se ha solicitado ningún FITS todavía. Un error 403 requiere revisar la cuenta y sus permisos; un 404, la ruta.


In [5]:
def resolver_columnas(df, configuracion):
    alias = {'ra': ['ra', 'ra_deg', 'raj2000'],
             'dec': ['dec', 'dec_deg', 'dej2000'],
             'label': ['label', 'labels', 'etiqueta', 'class', 'target', 'ring_label']}
    resultado = {}
    for campo, opciones in alias.items():
        explicita = configuracion.get(campo)
        candidatas = ([explicita] if explicita is not None else
                      [c for c in df.columns if str(c).strip().lower() in opciones])
        if len(candidatas) != 1 or candidatas[0] not in df.columns:
            raise ValueError(f'Configura COLUMNAS[{campo!r}]. Candidatas: {candidatas}')
        resultado[campo] = candidatas[0]
    if len(set(resultado.values())) != 3:
        raise ValueError('RA, DEC y etiqueta deben ser columnas diferentes.')
    return resultado

def preparar_catalogo(df, columnas):
    tabla = pd.DataFrame({k: pd.to_numeric(df[v], errors='coerce')
                          for k, v in columnas.items()})
    tabla['fila_csv'] = np.arange(len(df)) + 2  # Encabezado en línea 1, sin saltos internos.
    valido = (np.isfinite(tabla['ra']) & np.isfinite(tabla['dec']) &
              tabla['ra'].ge(0) & tabla['ra'].lt(360) &
              tabla['dec'].between(-90, 90) & tabla['label'].isin(ETIQUETAS))
    excluidas = tabla.loc[~valido].copy()
    elegibles = tabla.loc[valido].copy()
    elegibles['label'] = elegibles['label'].astype(int)
    conflictos = elegibles.groupby(['ra', 'dec'])['label'].nunique()
    if (conflictos > 1).any():
        raise ValueError('Hay coordenadas idénticas con etiquetas contradictorias; revisar catálogo.')
    duplicadas = int(elegibles.duplicated(['ra', 'dec']).sum())
    elegibles = elegibles.drop_duplicates(['ra', 'dec']).copy()
    elegibles['objeto_id'] = [sha(f'{ra:.17g},{dec:.17g}'.encode())[:24]
                             for ra, dec in zip(elegibles.ra, elegibles.dec)]
    return elegibles, excluidas, duplicadas

def seleccionar_piloto(tabla, semilla=SEMILLA):
    # Una plaza por estrato; restantes proporcionales a la capacidad residual.
    grupos = {e: tabla[tabla.label.eq(e)].copy() for e in ETIQUETAS}
    n = np.array([len(grupos[e]) for e in ETIQUETAS], dtype=int)
    if (n == 0).any() or n.sum() < N_PILOTO:
        raise ValueError(f'Faltan estratos o galaxias suficientes: {dict(zip(ETIQUETAS, n))}')
    capacidad = n - 1
    restantes = N_PILOTO - len(ETIQUETAS)
    ideales = restantes * capacidad / capacidad.sum()
    extra = np.floor(ideales).astype(int)
    orden = sorted(range(len(n)), key=lambda i: (-(ideales[i] - extra[i]), ETIQUETAS[i]))
    for i in orden[:restantes - int(extra.sum())]:
        extra[i] += 1
    cuotas = 1 + extra
    seleccion = []
    for etiqueta, cuota in zip(ETIQUETAS, cuotas):
        grupo = grupos[etiqueta].copy()
        # Ranking hash determinista: independiente del orden de las filas y de NumPy RNG.
        grupo['ranking'] = [sha(f'{semilla}|{x}'.encode()) for x in grupo.objeto_id]
        seleccion.append(grupo.sort_values(['ranking', 'objeto_id']).head(int(cuota)))
    piloto = pd.concat(seleccion, ignore_index=True).drop(columns='ranking')
    assert len(piloto) == 10 and piloto.objeto_id.nunique() == 10
    return piloto

columnas = resolver_columnas(catalogo, COLUMNAS)
elegibles, excluidas, duplicadas = preparar_catalogo(catalogo, columnas)
piloto = seleccionar_piloto(elegibles)
print('Mapeo:', columnas, '| Excluidas:', len(excluidas), '| Duplicados exactos:', duplicadas)
display(piloto)
display(piloto.groupby('label').size().rename('seleccionadas').to_frame()
        .join(elegibles.groupby('label').size().rename('disponibles')))

Mapeo: {'ra': 'ra', 'dec': 'dec', 'label': 'anillos'} | Excluidas: 0 | Duplicados exactos: 0


,ra,dec,label,fila_csv,objeto_id
0,182.359000,44.073299,0,2690,989ce52431e11ed67756e0df
1,248.426410,39.185141,0,1119,c2702d018d0ccb9e3252bf1b
2,235.349590,20.042084,0,6452,f55adb996c8de6ebc705eec2
3,166.791560,28.425614,0,4932,ca927fc596369d514b099798
4,246.842090,39.109627,2,8388,efeaebeba25569009928b857
5,157.970260,13.518171,4,7368,ed606600ed0ecf33940d5f17
6,202.630320,12.519520,4,7965,e6e76b0190a297e4002692c8
7,156.857150,35.908675,8,7349,91618f9899382fa0b6535070
8,233.801990,5.063002,12,8268,62c1d4330f2f6a784099c54b
9,55.071009,1.094029,16,6780,406ff122211f44db15b75583


,seleccionadas,disponibles
label,,
0,4,6659
2,1,111
4,2,857
8,1,186
12,1,372
16,1,342


**Explicación.** Detecta nombres comunes, pero se detiene si falta una columna o hay varias candidatas. Convierte RA/DEC a números y exige grados válidos; no interpreta coordenadas sexagesimales ni adivina unidades. Descarta filas no válidas o fuera de los seis códigos, registra su número y evita duplicados con coordenadas exactamente iguales. No hace asociación espacial de posiciones cercanas.

La estratificación reserva una plaza por etiqueta y distribuye cuatro adicionales mediante restos mayores, ponderados por disponibilidad residual; los empates favorecen el código menor. Es una muestra para comprobar la tubería, no para estimar prevalencias. La elección dentro de cada estrato usa SHA-256 con semilla, coordenadas e identificador estable; se conserva la fila de origen para trazabilidad.


In [6]:
config_ciencia = {'version_pipeline': 1, 'catalogo': CATALOGO,
    'catalogo_sha256': sha_catalogo, 'columnas': columnas, 'semilla': SEMILLA,
    'n': N_PILOTO, 'etiquetas': ETIQUETAS, 'capa': CAPA, 'bandas': BANDAS,
    'tamano': TAMANO, 'escala': ESCALA, 'endpoint': ENDPOINT,
    'max_no_finita': MAX_FRACCION_NO_FINITA, 'tol_centro': TOL_CENTRO_ARCSEC}
PILOTO_ID = sha(serializar(config_ciencia))[:24]
EJECUCION_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S') + '-' + uuid.uuid4().hex[:12]
BASE_META = PREFIJO_META + PILOTO_ID + '/'
BASE_RAW = PREFIJO_RAW + PILOTO_ID + '/'

# Toda escritura es creación exclusiva; nunca se reemplaza un objeto existente.
def crear_inmutable(nombre, contenido, tipo='application/json'):
    blob = bucket.blob(nombre)
    try:
        blob.upload_from_string(contenido, content_type=tipo,
                                if_generation_match=0, timeout=120)
    except PreconditionFailed:
        blob.reload()
        existente = blob.download_as_bytes(if_generation_match=int(blob.generation), timeout=120)
        if sha(existente) != sha(contenido):
            raise RuntimeError(f'Conflicto: objeto existente diferente: {nombre}')
    return blob

seleccion_bytes = piloto.to_csv(index=False, float_format='%.17g').encode('utf-8')
crear_inmutable(BASE_META + 'configuracion.json', serializar(config_ciencia))
crear_inmutable(BASE_META + 'seleccion.csv', seleccion_bytes, 'text/csv')
versiones = {p: version(p) for p in ['numpy', 'pandas', 'astropy', 'matplotlib',
                                    'requests', 'google-cloud-storage']}
crear_inmutable(BASE_META + f'ejecuciones/{EJECUCION_ID}.json', serializar({
    'inicio_utc': ahora(), 'python': platform.python_version(), 'versiones': versiones,
    'catalogo_generacion': generacion_catalogo, 'filas': len(catalogo),
    'filas_excluidas': len(excluidas), 'duplicados_exactos': duplicadas,
    'http_intentos': INTENTOS_HTTP, 'timeout': TIMEOUT, 'max_bytes': MAX_BYTES}))
print('Piloto:', PILOTO_ID)
print('Selección persistente:', f'gs://{BUCKET}/{BASE_META}seleccion.csv')


Piloto: 395d457f6ab201747d7fba0e
Selección persistente: gs://galmorphai-science/metadata/legacy_surveys/pilot/395d457f6ab201747d7fba0e/seleccion.csv


**Explicación.** El identificador agrupa catálogo, parámetros y versión de la tubería. Cambiarlos crea un piloto separado; reejecutarlos idénticos recupera el mismo. La selección se guarda antes de descargar imágenes. Las escrituras usan `if_generation_match=0`, una precondición atómica que impide sobrescribir incluso con ejecuciones simultáneas. Si el objeto ya existe, sus bytes deben coincidir. Los registros de ejecución son únicos e inmutables.


In [7]:
def inspeccionar_fits(contenido, ra, dec):
    diagnostico = {'hdus': [], 'bandas': [], 'avisos': [], 'calidad': 'revisar'}
    with warnings.catch_warnings(record=True) as avisos:
        warnings.simplefilter('always')
        with fits.open(io.BytesIO(contenido), memmap=False, checksum=True) as hdus:
            hdus.verify('exception')
            for i, hdu in enumerate(hdus):
                diagnostico['hdus'].append({'indice': i, 'nombre': hdu.name,
                    'tipo': type(hdu).__name__, 'shape': list(hdu.data.shape) if hdu.data is not None else None})
            imagenes = [h for h in hdus if h.data is not None and h.data.ndim == 3]
            if len(imagenes) != 1:
                raise ValueError('Se esperaba un único cubo FITS; revisar estructura HDU.')
            hdu = imagenes[0]
            cubo = np.asarray(hdu.data, dtype=float)
            if cubo.shape != (3, TAMANO, TAMANO):
                raise ValueError(f'Dimensiones inesperadas: {cubo.shape}')
            cab = hdu.header
            bandas = [str(cab.get(f'BAND{i}', '')).strip().lower() for i in range(3)]
            diagnostico['bandas_cabecera'] = bandas
            if bandas != list(BANDAS):
                # No asignar colores ni nombres por el orden solicitado si falta evidencia FITS.
                raise ValueError(f'BAND0..2 no confirman grz: {bandas}')
            diagnostico['unidad'] = cab.get('BUNIT', 'no declarada')
            wcs = WCS(cab).celestial
            if not wcs.has_celestial:
                raise ValueError('El FITS no contiene una WCS celeste utilizable.')
            x, y = wcs.world_to_pixel(SkyCoord(ra*u.deg, dec*u.deg, frame='icrs'))
            centro = wcs.pixel_to_world((TAMANO-1)/2, (TAMANO-1)/2)
            separacion = float(centro.separation(SkyCoord(ra*u.deg, dec*u.deg)).arcsec)
            diagnostico['wcs'] = {'ctype': list(wcs.wcs.ctype), 'pixel_objetivo': [float(x), float(y)],
                                  'separacion_centro_arcsec': separacion}
            if not np.isfinite([x, y, separacion]).all():
                raise ValueError('La transformación WCS no es finita.')
            if not (-0.5 <= x < TAMANO-0.5 and -0.5 <= y < TAMANO-0.5):
                diagnostico['avisos'].append('Coordenada objetivo fuera del recorte')
            if separacion > TOL_CENTRO_ARCSEC:
                diagnostico['avisos'].append('Centro WCS alejado de la coordenada solicitada')
            for banda, plano in zip(bandas, cubo):
                finitos = plano[np.isfinite(plano)]
                fraccion = float(1 - finitos.size / plano.size)
                estad = {'banda': banda, 'fraccion_no_finita': fraccion,
                         'n_nan': int(np.isnan(plano).sum()), 'n_inf': int(np.isinf(plano).sum()),
                         'fraccion_cero': float(np.mean(plano == 0))}
                if finitos.size:
                    estad.update(dict(zip(['p01', 'mediana', 'p99'],
                                          [float(v) for v in np.percentile(finitos, [1, 50, 99])])))
                    estad['std'] = float(np.std(finitos))
                if not finitos.size or np.ptp(finitos) == 0:
                    diagnostico['avisos'].append(f'{banda}: sin señal variable finita')
                if fraccion > MAX_FRACCION_NO_FINITA:
                    diagnostico['avisos'].append(f'{banda}: exceso de píxeles no finitos')
                if estad['fraccion_cero'] > 0.5:
                    diagnostico['avisos'].append(f'{banda}: más de 50% de ceros; posible falta de cobertura')
                diagnostico['bandas'].append(estad)
        diagnostico['avisos'].extend(str(a.message) for a in avisos)
    diagnostico['calidad'] = 'revisar' if diagnostico['avisos'] else 'sin_alertas_basicas'
    return diagnostico


**Explicación.** Inspecciona todos los HDU, exige el cubo esperado y verifica las bandas con `BAND0..2`, sin deducirlas por la petición HTTP. Comprueba WCS, posición objetivo, separación del centro, NaN, infinitos, ceros, dispersión y percentiles. Si el servicio cambia su formato, conserva el archivo y registra el fallo para revisión. Las advertencias FITS se incluyen en el diagnóstico.

«Sin alertas básicas» no garantiza calidad científica. No se mide seeing, profundidad, saturación, contaminación por estrellas ni relación señal/ruido; estos cutouts de imagen no incluyen necesariamente mapas de peso o máscaras. Los ceros son sólo una señal heurística de cobertura, y no se sustituyen ni se descartan.


In [8]:
sesion_http = requests.Session()
sesion_http.headers.update({'User-Agent': 'GalMorphAI-pilot/1.0 (10-object scientific pilot)'})

def descargar_cutout(ra, dec):
    parametros = {'ra': ra, 'dec': dec, 'layer': CAPA, 'bands': BANDAS,
                  'size': TAMANO, 'pixscale': ESCALA}
    url = requests.Request('GET', ENDPOINT, params=parametros).prepare().url
    historial = []
    for intento in range(1, INTENTOS_HTTP + 1):
        estado = None
        try:
            with sesion_http.get(url, timeout=TIMEOUT, stream=True) as respuesta:
                estado = respuesta.status_code
                respuesta.raise_for_status()
                partes, longitud = [], 0
                for parte in respuesta.iter_content(65536):
                    longitud += len(parte)
                    if longitud > MAX_BYTES:
                        raise ValueError('Respuesta mayor que MAX_BYTES')
                    partes.append(parte)
                contenido = b''.join(partes)
                if not contenido.startswith(b'SIMPLE  ='):
                    raise ValueError('La respuesta no tiene firma FITS primaria; posible HTML o falta de cobertura')
                historial.append({'intento': intento, 'http': estado, 'resultado': 'recibido'})
                return contenido, {'url': url, 'url_final': respuesta.url,
                    'content_type': respuesta.headers.get('Content-Type'), 'intentos': historial}
        except (requests.RequestException, ValueError) as exc:
            historial.append({'intento': intento, 'http': estado, 'error': str(exc)[:1500]})
            if estado is not None and 400 <= estado < 500 and estado not in (408, 429):
                break
            if intento < INTENTOS_HTTP:
                time.sleep(min(2 ** intento, 30))
    error = RuntimeError('No se obtuvo un FITS válido tras los intentos permitidos')
    error.detalle = {'url': url, 'intentos': historial}
    raise error

LOCAL_EVENTOS = DIRECTORIO / f'eventos-{PILOTO_ID}-{EJECUCION_ID}.jsonl'

def registrar_evento(evento):
    evento = dict(evento, fecha_utc=ahora(), ejecucion=EJECUCION_ID, piloto=PILOTO_ID)
    contenido = serializar(evento)
    # Primero una copia local; si GCS falla, detener para evitar trabajo sin trazabilidad.
    with LOCAL_EVENTOS.open('ab') as archivo:
        archivo.write(contenido + b'\n')
    nombre = BASE_META + f'eventos/{evento["objeto_id"]}/{uuid.uuid4().hex}.json'
    crear_inmutable(nombre, contenido)
    return evento

def obtener_original(nombre, fila):
    blob = bucket.get_blob(nombre)
    if blob is not None:
        contenido = blob.download_as_bytes(if_generation_match=int(blob.generation), timeout=120)
        esperado = (blob.metadata or {}).get('sha256')
        if esperado is not None and sha(contenido) != esperado:
            raise ValueError('SHA-256 del original almacenado no coincide')
        return contenido, {'origen': 'gcs_reanudado', 'generacion': int(blob.generation),
                            'sha256': sha(contenido), 'sha256_previo_disponible': esperado is not None}
    contenido, red = descargar_cutout(float(fila['ra']), float(fila['dec']))
    blob = bucket.blob(nombre)
    blob.metadata = {'sha256': sha(contenido), 'url': red['url'], 'descarga_utc': ahora()}
    try:
        blob.upload_from_string(contenido, content_type='application/fits',
                                if_generation_match=0, timeout=120)
    except PreconditionFailed:
        # Otro proceso terminó primero. Usar su original, nunca reemplazarlo.
        return obtener_original(nombre, fila)
    return contenido, dict(red, origen='legacy_surveys', sha256=sha(contenido),
                            generacion=int(blob.generation))


**Explicación.** Las peticiones se hacen en serie, con tiempos máximos, límite de bytes y hasta tres intentos. Reintenta errores de red, respuestas incompletas y errores transitorios; los 4xx permanentes se registran sin insistir. La firma FITS evita guardar una página HTML como imagen. Los FITS con firma pero problemas internos sí se conservan para auditoría.

Antes de contactar Legacy Surveys, consulta GCS. Si existe el original, recupera su generación y verifica su hash cuando está disponible. Cada evento tiene un objeto único en GCS y una copia JSONL local. Una caída del registro persistente detiene el procesamiento; al reiniciar se puede recuperar un FITS subido aunque faltara su evento final.


In [9]:
assert len(piloto) == 10 and piloto.objeto_id.nunique() == 10
resultados = []
for fila in piloto.to_dict(orient='records'):
    objeto_id = fila['objeto_id']
    nombre = BASE_RAW + objeto_id + '.fits'
    base = dict(fila, raw_uri=f'gs://{BUCKET}/{nombre}')
    registrar_evento(dict(base, estado='iniciado'))
    transferencia = {}
    try:
        contenido, transferencia = obtener_original(nombre, fila)
        local = DIRECTORIO / f'{PILOTO_ID}-{objeto_id}.fits'
        local.write_bytes(contenido)  # Sólo caché local; GCS es el original persistente.
        diagnostico = inspeccionar_fits(contenido, float(fila['ra']), float(fila['dec']))
        final = dict(base, estado='completado', transferencia=transferencia,
                     diagnostico=diagnostico, bytes=len(contenido))
    except Exception as exc:
        final = dict(base, estado='error', transferencia=transferencia,
                     tipo_error=type(exc).__name__, error=str(exc)[:3000],
                     detalle=getattr(exc, 'detalle', None))
    # Fuera del try: un fallo al persistir no queda oculto como error de una galaxia.
    resultados.append(registrar_evento(final))
    print(objeto_id, '| etiqueta', fila['label'], '|', final['estado'])
    time.sleep(PAUSA_SEGUNDOS)
print('Finalizó el recorrido de los diez objetos. Consultar errores y alertas abajo.')


989ce52431e11ed67756e0df | etiqueta 0 | completado
c2702d018d0ccb9e3252bf1b | etiqueta 0 | completado
f55adb996c8de6ebc705eec2 | etiqueta 0 | completado
ca927fc596369d514b099798 | etiqueta 0 | completado
efeaebeba25569009928b857 | etiqueta 2 | completado
ed606600ed0ecf33940d5f17 | etiqueta 4 | completado
e6e76b0190a297e4002692c8 | etiqueta 4 | completado
91618f9899382fa0b6535070 | etiqueta 8 | completado
62c1d4330f2f6a784099c54b | etiqueta 12 | completado
406ff122211f44db15b75583 | etiqueta 16 | completado
Finalizó el recorrido de los diez objetos. Consultar errores y alertas abajo.


**Explicación.** Ésta es la única celda que descarga el piloto. Registra inicio y resultado por galaxia y continúa ante fallos individuales. Un error de autenticación global puede producir errores por objeto; corrige permisos antes de reejecutar. No reemplaza una galaxia fallida por otra: preserva la muestra seleccionada.

Al reanudar vuelve a inspeccionar los originales presentes, sin descargarlos otra vez desde Legacy Surveys. Los fallos de red sin original se reintentan; un FITS persistido pero defectuoso vuelve a señalarse para revisión y no se sobrescribe. Una ejecución interrumpida puede dejar un evento `iniciado`; la siguiente añadirá nuevos eventos y recuperará el estado.


In [10]:
# Reconstrucción del manifiesto desde eventos persistentes, incluso tras reiniciar Colab.
eventos = []
for blob in cliente.list_blobs(BUCKET, prefix=BASE_META + 'eventos/'):
    eventos.append(json.loads(blob.download_as_bytes(
        if_generation_match=int(blob.generation), timeout=120)))
ultimos = {}
for evento in sorted(eventos, key=lambda e: (e['fecha_utc'], e['ejecucion'])):
    ultimos[evento['objeto_id']] = evento
filas_resumen = []
for fila in piloto.to_dict(orient='records'):
    ev = ultimos.get(fila['objeto_id'], {})
    filas_resumen.append(dict(fila, estado=ev.get('estado', 'pendiente'),
        calidad=ev.get('diagnostico', {}).get('calidad'),
        error=ev.get('error'), raw_uri=ev.get('raw_uri'),
        sha256=ev.get('transferencia', {}).get('sha256'),
        fecha_utc=ev.get('fecha_utc')))
resumen = pd.DataFrame(filas_resumen)
manifest = {'piloto': PILOTO_ID, 'configuracion': config_ciencia,
            'seleccion': piloto.to_dict(orient='records'), 'ultimos_eventos': ultimos,
            'numero_eventos': len(eventos), 'creado_utc': ahora()}
SNAPSHOT_ID = uuid.uuid4().hex
nombre_manifest = BASE_META + f'manifiestos/{SNAPSHOT_ID}.json'
crear_inmutable(nombre_manifest, serializar(manifest))
crear_inmutable(BASE_META + f'manifiestos/{SNAPSHOT_ID}.csv',
                 resumen.to_csv(index=False).encode('utf-8'), 'text/csv')
(DIRECTORIO / 'manifiesto.json').write_bytes(serializar(manifest))
resumen.to_csv(DIRECTORIO / 'resumen.csv', index=False)
display(resumen)
print('Manifiesto completo:', f'gs://{BUCKET}/{nombre_manifest}')
print('Estados:', resumen.estado.value_counts().to_dict())


,ra,dec,label,fila_csv,objeto_id,estado,calidad,error,raw_uri,sha256,fecha_utc
0,182.359000,44.073299,0,2690,989ce52431e11ed67756e0df,completado,revisar,None,gs://galmorphai-science/raw/legacy_surveys/pil...,b9ea59aca4329cddca2b81c55d91f90a34bce382f46427...,2026-10-15T23:47:22.452687+00:00
1,248.426410,39.185141,0,1119,c2702d018d0ccb9e3252bf1b,completado,revisar,None,gs://galmorphai-science/raw/legacy_surveys/pil...,4682849ae099a2bf7c1c5247681f6cfdd18653eced9f42...,2026-10-15T23:47:24.867511+00:00
2,235.349590,20.042084,0,6452,f55adb996c8de6ebc705eec2,completado,revisar,None,gs://galmorphai-science/raw/legacy_surveys/pil...,3c856be1e32b9f426b0e38989f66e32cff045927da5102...,2026-10-15T23:47:34.489068+00:00
3,166.791560,28.425614,0,4932,ca927fc596369d514b099798,completado,revisar,None,gs://galmorphai-science/raw/legacy_surveys/pil...,bdb8c0fcfbcf161e896dc5d0ccffc8f1fe6792e40764cf...,2026-10-15T23:47:37.205980+00:00
4,246.842090,39.109627,2,8388,efeaebeba25569009928b857,completado,revisar,None,gs://galmorphai-science/raw/legacy_surveys/pil...,e28269413702242d6b3fe15b75a71d8b109b2a1a119f96...,2026-10-15T23:47:39.131240+00:00
5,157.970260,13.518171,4,7368,ed606600ed0ecf33940d5f17,completado,revisar,None,gs://galmorphai-science/raw/legacy_surveys/pil...,755541fca04c1a3e83418f454ae3caf88dd617f9ed5223...,2026-10-15T23:47:42.015169+00:00
6,202.630320,12.519520,4,7965,e6e76b0190a297e4002692c8,completado,revisar,None,gs://galmorphai-science/raw/legacy_surveys/pil...,c7c0798a423fd75cc6d2f7527085c68b6d7271b9bf8762...,2026-10-15T23:47:44.213143+00:00
7,156.857150,35.908675,8,7349,91618f9899382fa0b6535070,completado,revisar,None,gs://galmorphai-science/raw/legacy_surveys/pil...,304f0008e02af5e092a466531c450587ad8353ba6432c7...,2026-10-15T23:47:46.270366+00:00
8,233.801990,5.063002,12,8268,62c1d4330f2f6a784099c54b,completado,revisar,None,gs://galmorphai-science/raw/legacy_surveys/pil...,71d05b47bba10df088700323d690248cd9172121a6fe58...,2026-10-15T23:47:48.487100+00:00
9,55.071009,1.094029,16,6780,406ff122211f44db15b75583,completado,revisar,None,gs://galmorphai-science/raw/legacy_surveys/pil...,cb942261e1824d5b86313122887fca77b21b249762f05c...,2026-10-15T23:47:50.741433+00:00


Manifiesto completo: gs://galmorphai-science/metadata/legacy_surveys/pilot/395d457f6ab201747d7fba0e/manifiestos/5029c13a402e4b96afa78b1d4a10a0d9.json
Estados: {'completado': 10}


**Explicación.** El manifiesto completo incluye coordenadas, códigos, rutas, hashes, parámetros, diagnósticos y errores; el CSV es una vista resumida. Cada reconstrucción crea una instantánea nueva. Los eventos anteriores permanecen disponibles para auditar todos los intentos. El estado mostrado corresponde al evento más reciente por objeto; si es `iniciado`, la ejecución quedó pendiente o sigue activa. Evita ejecutar dos sesiones del mismo piloto simultáneamente para facilitar la interpretación temporal.


In [11]:
def estirar_visual(plano):
    finitos = plano[np.isfinite(plano)]
    if not finitos.size:
        return np.zeros_like(plano, dtype=float)
    bajo, alto = np.percentile(finitos, [1, 99])
    if alto <= bajo:
        return np.zeros_like(plano, dtype=float)
    return np.arcsinh(10 * np.clip((np.nan_to_num(plano, nan=bajo,
                    posinf=alto, neginf=bajo) - bajo) / (alto-bajo), 0, 1)) / np.arcsinh(10)

for fila in piloto.to_dict(orient='records'):
    ev = ultimos.get(fila['objeto_id'], {})
    if ev.get('estado') != 'completado':
        print('Sin visualización validada:', fila['objeto_id'], ev.get('estado', 'pendiente'))
        continue
    nombre = BASE_RAW + fila['objeto_id'] + '.fits'
    blob = bucket.get_blob(nombre)
    contenido = blob.download_as_bytes(if_generation_match=int(blob.generation), timeout=120)
    if sha(contenido) != ev['transferencia']['sha256']:
        raise ValueError('El hash cambió antes de visualizar')
    with fits.open(io.BytesIO(contenido), memmap=False) as hdus:
        hdu = next(h for h in hdus if h.data is not None and h.data.ndim == 3)
        cubo = hdu.data.astype(float)
    fig, axes = plt.subplots(1, 4, figsize=(13, 3))
    for i, banda in enumerate(BANDAS):
        axes[i].imshow(np.ma.masked_invalid(cubo[i]), origin='lower', cmap='gray',
                       vmin=ev['diagnostico']['bandas'][i].get('p01'),
                       vmax=ev['diagnostico']['bandas'][i].get('p99'))
        axes[i].set_title(banda)
    # Composición ilustrativa: z -> rojo, r -> verde, g -> azul.
    rgb = np.dstack([estirar_visual(cubo[i]) for i in [2, 1, 0]])
    axes[3].imshow(rgb, origin='lower')
    axes[3].set_title('RGB ilustrativo z/r/g')
    for ax in axes:
        ax.set_axis_off()
    fig.suptitle(f"{fila['objeto_id']} · etiqueta {fila['label']} · "
                 + ev['diagnostico']['calidad'])
    fig.tight_layout()
    plt.show()


Output hidden; open in https://colab.research.google.com to view.

**Explicación.** Muestra las bandas por separado y una composición RGB ilustrativa con estiramiento independiente por banda. El reemplazo de valores no finitos y el estiramiento ocurren sólo en copias para mostrar: no cambian los FITS originales. Los colores no son una calibración fotométrica ni equivalen al renderizado oficial del visor. Los píxeles NaN permanecen enmascarados en las vistas monocromáticas. Revisa cobertura, centrado, bordes y fuentes vecinas.


In [12]:
# Pruebas autocontenidas: sin credenciales, sin GCS y sin descargas.
def pruebas_locales():
    datos = pd.DataFrame([{'ra': 10 + e + j/100, 'dec': -5 + j/10,
                          'label': e, 'objeto_id': f'{e}-{j}', 'fila_csv': j+2}
                         for e in ETIQUETAS for j in range(4)])
    a = seleccionar_piloto(datos)
    b = seleccionar_piloto(datos.sample(frac=1, random_state=9))
    assert a.objeto_id.tolist() == b.objeto_id.tolist()
    assert len(a) == 10 and set(a.label) == set(ETIQUETAS)
    try:
        seleccionar_piloto(datos[datos.label.ne(16)])
    except ValueError:
        pass
    else:
        raise AssertionError('Debe fallar si falta una etiqueta')
    conf = pd.DataFrame({'ra': [1, 1], 'dec': [2, 2], 'label': [0, 2]})
    try:
        preparar_catalogo(conf, {'ra':'ra', 'dec':'dec', 'label':'label'})
    except ValueError:
        pass
    else:
        raise AssertionError('Debe rechazar etiquetas contradictorias')
    w = WCS(naxis=2)
    w.wcs.crpix = [(TAMANO+1)/2, (TAMANO+1)/2]
    w.wcs.cdelt = [-ESCALA/3600, ESCALA/3600]
    w.wcs.crval = [120, 10]
    w.wcs.ctype = ['RA---TAN', 'DEC--TAN']
    h = w.to_header()
    for i, banda in enumerate(BANDAS):
        h[f'BAND{i}'] = banda
    datos_fits = np.random.default_rng(42).normal(size=(3, TAMANO, TAMANO)).astype('float32')
    datos_fits[0, 0, 0] = np.nan
    buffer = io.BytesIO()
    fits.PrimaryHDU(datos_fits, header=h).writeto(buffer)
    diag = inspeccionar_fits(buffer.getvalue(), 120, 10)
    assert diag['bandas'][0]['n_nan'] == 1
    assert diag['wcs']['separacion_centro_arcsec'] < 0.01
    assert diag['calidad'] == 'sin_alertas_basicas'
    print('Pruebas locales: OK (muestreo, estratos, conflictos, FITS, NaN y WCS).')

pruebas_locales()


Pruebas locales: OK (muestreo, estratos, conflictos, FITS, NaN y WCS).


**Explicación.** Las pruebas usan un catálogo y un FITS sintéticos. Comprueban la reproducibilidad ante cambios de orden, la representación de todas las etiquetas, la parada ante conflictos y la detección de NaN con WCS consistente. No demuestran que la cuenta tenga permisos ni que las diez posiciones del catálogo real tengan cobertura. Puedes ejecutar esta celda tras definir las funciones, antes de procesar el piloto.


## Reanudación y diagnóstico de fallos

- **403 de GCS:** comprueba la cuenta autenticada y permisos de lectura/listado/creación. No se cambia IAM desde este notebook.
- **Columnas ambiguas:** edita `COLUMNAS` con los nombres exactos y vuelve a ejecutar desde selección.
- **Estrato ausente o menos de diez objetos elegibles:** se detiene; revisa el catálogo, sin inventar etiquetas ni duplicar galaxias.
- **HTTP 429, 5xx o timeout:** los intentos son acotados. Reejecuta después; cada fallo queda en el historial.
- **FITS sin cobertura, bandas inesperadas o WCS inválida:** revisa el evento y el original. No se reemplaza el objeto persistido. Para una nueva petición con otros parámetros, cambia explícitamente capa, tamaño o escala y se creará otro identificador de piloto.
- **Fallo al guardar eventos:** se detiene. La copia JSONL queda en `/content/galmorphai_pilot`, pero Colab puede perder ese almacenamiento al cerrar el entorno. Descárgala antes de cerrar si GCS continúa inaccesible.
- **Interrupción:** con catálogo y configuración iguales, los originales en GCS permiten continuar. Si el catálogo cambió, esta ejecución tendrá otro identificador; para reproducir la anterior necesitas los mismos bytes del catálogo o recuperar su versión en GCS si está disponible.

## Alcance de validación y siguientes decisiones

Esta versión conserva las salidas de la ejecución del usuario en Colab: diez objetos completados y diez con calidad pendiente de revisión. La corrección posterior centraliza el mapeo de columnas y no vuelve a ejecutar la descarga. Consulta `VALIDACION.md` junto al notebook para distinguir las pruebas realizadas de las verificaciones pendientes.

El piloto no prueba rendimiento de un modelo ni autoriza procesar el catálogo completo. Antes de ampliar el trabajo, revisa manualmente los diez recortes y define criterios científicos de calidad, escala angular y tratamiento posterior de píxeles no finitos.

## Fuentes técnicas

- [Legacy Surveys DR10: descarga de imágenes y parámetros del visor](https://www.legacysurvey.org/dr10/description/).
- [Google Cloud Storage: precondiciones y creación exclusiva](https://docs.cloud.google.com/storage/docs/request-preconditions).
- [Ejemplo oficial de autenticación y datos externos en Colab](https://colab.research.google.com/notebooks/io.ipynb).

La capa DR10 se fija para trazabilidad; no se selecciona automáticamente la entrega más reciente. La documentación de DR10 indica la combinación de imágenes del norte y del sur. Las condiciones de cobertura deben comprobarse por objeto.
